# Customer Analytics & Campaign Intelligence — Exploratory Notebook

This notebook mirrors the data pipeline used in the Streamlit dashboard.
It is intended for:
- Experimentation and EDA
- Verifying pipeline steps interactively
- Interview demonstrations

**Dataset:** UCI Online Retail II (2010-2011)  
**Source:** https://archive.ics.uci.edu/dataset/502/online+retail+ii  
**Licence:** CC BY 4.0

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

from data_loader import download_dataset, load_raw_data, load_to_sqlite, run_sql
from data_loader import SQL_REVENUE_BY_MONTH, SQL_TOP_CUSTOMERS, SQL_TOP_PRODUCTS
from preprocessing import validate_raw, clean_data
from rfm import compute_rfm, add_rfm_scores
from segmentation import scale_rfm, compute_elbow_silhouette, choose_k, train_kmeans, assign_segments
from campaign_model import build_features_target, train_model, evaluate_model, score_all_customers

print('Imports OK')

## 1. Load Raw Data

In [ ]:
xlsx = download_dataset()
raw = load_raw_data(xlsx)
print(f'Shape: {raw.shape}')
raw.head()

## 2. Data Validation

In [ ]:
quality = validate_raw(raw)
for k, v in quality.items():
    if isinstance(v, dict):
        print(f'\n{k}:')
        for col, val in v.items():
            print(f'  {col}: {val}')
    else:
        print(f'{k}: {v}')

## 3. Data Cleaning

In [ ]:
df, log = clean_data(raw)
print('Cleaning log:')
for k, v in log.items():
    print(f'  {k}: {v}')
df.head()

## 4. Load to SQLite & run SQL queries

In [ ]:
load_to_sqlite(df)
run_sql(SQL_REVENUE_BY_MONTH).head(12)

In [ ]:
run_sql(SQL_TOP_CUSTOMERS).head(10)

## 5. EDA — Revenue Over Time

In [ ]:
monthly = df.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly.columns = ['Month', 'Revenue']
px.line(monthly, x='Month', y='Revenue', title='Monthly Revenue', markers=True)

## 6. RFM Metrics

In [ ]:
rfm = compute_rfm(df)
rfm = add_rfm_scores(rfm)
print(rfm.shape)
rfm.describe().round(2)

## 7. K-Means Segmentation

In [ ]:
X, scaler, features = scale_rfm(rfm)
metrics = compute_elbow_silhouette(X, k_range=range(2, 9))

print('K  | Silhouette')
for k, s in zip(metrics['k'], metrics['silhouette']):
    print(f'{k}  | {s:.4f}')

best_k = choose_k(metrics)
print(f'\nChosen K = {best_k}')

In [ ]:
km = train_kmeans(X, best_k)
rfm_seg, stats, label_map = assign_segments(rfm, km.labels_)
print('Segment assignments:', label_map)
rfm_seg.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2)

## 8. Campaign Propensity Model

In [ ]:
X_feat, y, customers, feat_df = build_features_target(df)
print(f'Customers in model: {len(X_feat)}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
model, scaler_lr, X_tr, X_te, y_tr, y_te = train_model(X_feat, y)
metrics_lr = evaluate_model(model, X_te, y_te)

print('Model Metrics')
for k, v in metrics_lr.items():
    if k not in ('confusion_matrix', 'classification_report'):
        print(f'  {k}: {v}')

print('\nClassification Report')
print(metrics_lr['classification_report'])

In [ ]:
scored = score_all_customers(model, scaler_lr, X_feat, customers, rfm_seg)
scored.head(20)

## 9. Summary

This notebook walks through the full pipeline:
1. Raw data download and loading
2. Data quality validation
3. Data cleaning with documented decisions
4. SQL analysis via SQLite
5. EDA with Plotly
6. RFM metric computation
7. K-Means clustering with K selection
8. Logistic Regression propensity model

All steps are reproducible (random_state=42 throughout).
The Streamlit dashboard (`app.py`) runs the same pipeline with interactive charts.